# Click-Through Rate Prediction on the Avazu Dataset

**Author:** Mahmoud BAJJOU  

---

## Project Overview

Click-Through Rate (CTR) prediction is a core problem in computational advertising. Given the context of an ad impression (user device, site, time of day, etc.), we want to estimate the probability that a user will click on the ad.

This project explores the **Avazu dataset** — 1 million ad impressions collected over 11 days — and benchmarks several machine learning approaches:

| Step | Method | Key idea |
|------|---------|----------|
| 1 | **Logistic Regression** (reduced features) | One-Hot Encoding on 12 interpretable columns |
| 2 | **Logistic Regression** (full features + frequency filtering) | Keep only modalities seen ≥100 times |
| 3 | **GBDT + Logistic Regression** | Use tree leaves as non-linear features for LR |
| 4 | **XGBoost** | Optimized gradient boosting |
| 5 | **Feature Hashing + Random Forest** | Scalable encoding for high-cardinality features |

**Key challenge:** the dataset is heavily imbalanced (only ~17% clicks), so accuracy is a poor metric. We use **Log-Loss** and **ROC AUC** throughout.

## 1. Setup & Data Loading

In [ ]:
import sys
from zipfile import ZipFile
import os.path as op
try:
    from urllib.request import urlretrieve
except ImportError:
    from urllib import urlretrieve

AVAZU_URL = "https://bianchi.wp.imt.fr/files/2019/05/train-1000000.zip"
AVAZU_FILENAME = AVAZU_URL.rsplit('/', 1)[1]

if not op.exists(AVAZU_FILENAME):
    print(f'Downloading {AVAZU_URL}...')
    urlretrieve(AVAZU_URL, AVAZU_FILENAME)
    ZipFile(AVAZU_FILENAME).extractall('.')
    print('Done.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

## 2. Dataset Exploration

The Avazu dataset contains 1 million rows and 24 columns, including:
- `click` : binary target (1 = clicked, 0 = not clicked)
- `hour` : timestamp in compact format `YYMMDDHH`
- Categorical features describing the ad context: site, app, device, banner position, etc.

Full feature descriptions: https://www.kaggle.com/c/avazu-ctr-prediction/data

In [ ]:
df = pd.read_csv('train-1000000')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Class distribution — the dataset is strongly imbalanced
click_freq = df['click'].value_counts(normalize=True)
print(click_freq)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
click_freq.plot(kind='bar', ax=ax[0], color=['steelblue', 'tomato'], rot=0)
ax[0].set_title('Click vs No-Click frequency')
ax[0].set_ylabel('Proportion')

# A naive classifier that always predicts 0 would reach ~83% accuracy
# => accuracy is misleading here; we rely on Log-Loss and ROC AUC
ax[1].pie(click_freq.values, labels=['No click (83%)', 'Click (17%)'],
          colors=['steelblue', 'tomato'], autopct='%1.1f%%', startangle=90)
ax[1].set_title('Class imbalance')
plt.tight_layout()
plt.show()

In [ ]:
# Number of distinct values per feature
# Features like device_id (150k) or device_ip (555k) have very high cardinality
# => direct One-Hot Encoding would create ~1.7M features
cardinality = df.nunique().sort_values(ascending=False)
print(cardinality)
print(f'\nTotal dimension with naive OHE: {cardinality.values.sum():,}')

## 3. Feature Engineering

### 3.1 Datetime Parsing

The `hour` column encodes date and time as a single integer (e.g. `14102915` = 2014-10-29 15h).  
We extract:
- **`weekday`**: day of the week (0=Monday ... 6=Sunday)
- **`hour`**: hour of the day (0–23), which replaces the original column

In [ ]:
def datesplit(originalDate):
    """Parse compact YYMMDDHH integer into a datetime object."""
    s = str(originalDate)
    return datetime.datetime(
        year=int('20' + s[0:2]),
        month=int(s[2:4]),
        day=int(s[4:6]),
        hour=int(s[6:8])
    )

df['weekday'] = df['hour'].apply(lambda x: datesplit(x).weekday())
df['hour']    = df['hour'].apply(lambda x: datesplit(x).hour)

In [ ]:
# CTR varies with hour of day and day of week — useful signals for the model
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.groupby('hour')['click'].mean().plot(ax=axes[0], marker='o', color='steelblue')
axes[0].set_title('CTR by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Mean CTR')
axes[0].grid(alpha=0.4)

days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
ctr_by_day = df.groupby('weekday')['click'].mean()
axes[1].bar(days, ctr_by_day.values, color='tomato', alpha=0.8)
axes[1].set_title('CTR by Day of Week')
axes[1].set_ylabel('Mean CTR')
axes[1].grid(alpha=0.4, axis='y')

plt.tight_layout()
plt.show()

print('Observation: CTR peaks in the early morning (0-2h) and on weekends/Mondays.')

### 3.2 Device User Reconstruction

The `device_id` column is supposed to uniquely identify a user.  
However, Android devices that don't expose their real ID all share the same placeholder value `a99f21e6` — making it useless as-is (it aggregates millions of distinct users).

**Fix:** for that specific value, we rebuild a proxy user ID by concatenating `device_ip + device_model`, which is a reasonable approximation of uniqueness.

$$
\text{user} = \begin{cases} \text{device\_ip} + \text{device\_model} & \text{if device\_id = a99f21e6} \\ \text{device\_id} & \text{otherwise} \end{cases}
$$

In [ ]:
# Identify the generic Android placeholder ID
V = df['device_id'].value_counts().index[0]
print(f'Generic placeholder device_id: {V}')
print(f'Occurrences: {(df["device_id"] == V).sum():,} ({(df["device_id"] == V).mean():.1%} of data)')

# Reconstruct a more meaningful user identifier
df['user'] = (
    (df['device_ip'] + df['device_model']) * (df['device_id'] == V) +
    df['device_id'] * (df['device_id'] != V)
)

# Drop the redundant raw columns
df.drop(columns=['device_id', 'device_model', 'device_ip'], inplace=True)

### 3.3 Site Feature Consolidation

`site_id` and `site_domain` carry overlapping information (a site belongs to one domain). Merging them into a single `site` feature reduces redundancy while preserving the joint information.

In [ ]:
# Merge site_id and site_domain into a single composite key
df['site'] = df['site_id'] + df['site_domain']
df.drop(columns=['site_id', 'site_domain'], inplace=True)

print(f'Features after engineering: {df.shape[1]}')
print(df.columns.tolist())

### 3.4 CTR Analysis by Feature

Before building models, it helps to visualize which feature values are most predictive of clicks.

In [ ]:
# C15 = ad width in pixels — strong effect on CTR
ctr_c15 = df.groupby('C15')['click'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 4))
ctr_c15.plot(kind='bar', color='steelblue', alpha=0.8)
plt.title('CTR by Ad Width (C15)')
plt.xlabel('Ad width (pixels)')
plt.ylabel('Mean CTR')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

print(ctr_c15)

In [ ]:
# Site category: combine mean CTR and count to assess statistical reliability
# Low-count categories may show extreme CTR simply due to small sample size
col = 'site_category'
site_stats = pd.DataFrame({
    'mean_ctr': df.groupby(col)['click'].mean(),
    'count': df.groupby(col)['click'].count()
}).sort_values('count', ascending=False)

print(site_stats)
print('\nNote: categories with count < 100 have unreliable CTR estimates.')

## 4. Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, roc_auc_score, roc_curve, accuracy_score

# Drop 'id' — it is a unique row identifier, not a predictive feature
Xtrain, Xtest, ytrain, ytest = train_test_split(
    df.drop(['id', 'click'], axis=1),
    df['click'],
    test_size=0.1,
    random_state=100
)

print(f'Train size: {len(Xtrain):,} samples')
print(f'Test size:  {len(Xtest):,} samples')

In [ ]:
def evaluate_model(y_true, y_proba, model_name='Model'):
    """Compute and display Log-Loss, ROC AUC, and ROC curve."""
    ll  = log_loss(y_true, y_proba)
    auc = roc_auc_score(y_true, y_proba)
    print(f'[{model_name}]  Log-Loss: {ll:.5f}  |  ROC AUC: {auc:.5f}')

    fpr, tpr, _ = roc_curve(y_true, y_proba)
    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, label=f'{model_name} (AUC={auc:.3f})')
    plt.plot([0, 1], [0, 1], 'r--', label='Random')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve — {model_name}')
    plt.legend()
    plt.grid(alpha=0.4)
    plt.tight_layout()
    plt.show()
    return ll, auc

results = {}  # Store final metrics for comparison

## 5. Model 1 — Logistic Regression on Reduced Feature Set

We start with 12 columns that have **low cardinality** (at most a few hundred modalities), making One-Hot Encoding tractable.  
This gives us a dense but interpretable baseline with only **183 binary features**.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

# Low-cardinality features — safe for One-Hot Encoding
some_columns = [
    'hour', 'weekday', 'C1', 'banner_pos',
    'site_category', 'app_category',
    'device_type', 'device_conn_type',
    'C15', 'C16', 'C18', 'C21'
]

# Fit encoder on train, apply same transformation to test (no data leakage)
ohe_reduced = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
Xtrain_oh_r = ohe_reduced.fit_transform(Xtrain[some_columns])
Xtest_oh_r  = ohe_reduced.transform(Xtest[some_columns])

print(f'Number of features after OHE: {Xtrain_oh_r.shape[1]}')

In [ ]:
lr_reduced = LogisticRegression(max_iter=500, solver='lbfgs', C=1.0)
lr_reduced.fit(Xtrain_oh_r, ytrain)

soft_r = lr_reduced.predict_proba(Xtest_oh_r)[:, 1]
ll, auc = evaluate_model(ytest, soft_r, 'LR (reduced features)')
results['LR (reduced)'] = {'log_loss': ll, 'roc_auc': auc}

## 6. Model 2 — Logistic Regression on Full Features (Frequency Filtering)

Using all columns leads to **713,334 possible One-Hot features**, but the vast majority are seen only once or twice (e.g., rare device IDs, uncommon sites).  
Such features harm generalization (overfitting to rare events) and slow down training.

**Strategy:** keep only modalities seen **at least 100 times** in the training set → reduces to **1,971 features**.

In [ ]:
# Encode all features
ohe_full = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
Xtrain_oh = ohe_full.fit_transform(Xtrain)
Xtest_oh  = ohe_full.transform(Xtest)

print(f'Total OHE features (all columns): {Xtrain_oh.shape[1]:,}')

# Count training occurrences per modality (= column sum in sparse matrix)
n_ones = np.array(Xtrain_oh.sum(axis=0)).flatten()

# Keep only modalities with at least 100 training samples
FREQ_THRESHOLD = 100
useful = n_ones > FREQ_THRESHOLD
cols_to_keep = [i for i, b in enumerate(useful) if b]

print(f'Features seen ≥ {FREQ_THRESHOLD} times: {len(cols_to_keep):,}')

In [ ]:
lr_full = LogisticRegression(max_iter=1000, solver='lbfgs', C=1.0)
lr_full.fit(Xtrain_oh[:, cols_to_keep], ytrain)

soft_full = lr_full.predict_proba(Xtest_oh[:, cols_to_keep])[:, 1]
ll, auc = evaluate_model(ytest, soft_full, 'LR (full + freq. filter)')
results['LR (full)'] = {'log_loss': ll, 'roc_auc': auc}

## 7. Model 3 — Gradient Boosted Trees + Logistic Regression (GBDT-LR)

This is a classic technique introduced by Facebook in 2014 for CTR prediction:

1. Train a **Gradient Boosted Decision Tree** (GBDT) on the data.
2. For each tree, record in which **leaf** each sample falls → this is a non-linear feature transformation.
3. One-Hot encode these leaf indices and feed them into a **Logistic Regression**.

**Why does this work?** Each leaf represents a conjunction of conditions learnt by the tree (e.g., "hour=2 AND site_category=X AND device_type=mobile"). The LR then learns which conjunctions are predictive of clicks — capturing interaction effects that plain LR misses.


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Train the GBDT on the frequency-filtered OHE features
gb = GradientBoostingClassifier(
    n_estimators=50,
    learning_rate=0.8,
    max_depth=3,       # shallow trees reduce overfitting
    verbose=True
)
gb.fit(Xtrain_oh[:, cols_to_keep], ytrain)

In [ ]:
# Extract leaf indices: shape (n_samples, n_estimators)
# gb.apply returns shape (n_samples, n_estimators, 1) for binary classification
Xtrain_leaves = gb.apply(Xtrain_oh[:, cols_to_keep])[:, :, 0]
Xtest_leaves  = gb.apply(Xtest_oh[:, cols_to_keep])[:, :, 0]

print(f'Leaf feature matrix shape: {Xtrain_leaves.shape}  '
      f'({Xtrain_leaves.shape[1]} trees × each sample gets one leaf index)')

# One-Hot encode the leaf indices: each tree contributes 2^max_depth = 8 possible leaves
from scipy.sparse import hstack

ohe_leaves = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
leaves_train_oh = ohe_leaves.fit_transform(Xtrain_leaves)
leaves_test_oh  = ohe_leaves.transform(Xtest_leaves)

print(f'Leaf OHE features: {leaves_train_oh.shape[1]}')

# Concatenate leaf features with original OHE features
Xtrain_concat = hstack([leaves_train_oh, Xtrain_oh[:, cols_to_keep]])
Xtest_concat  = hstack([leaves_test_oh,  Xtest_oh[:, cols_to_keep]])

print(f'Final feature matrix: {Xtrain_concat.shape}')

In [ ]:
lr_gbdt = LogisticRegression(max_iter=1000, solver='lbfgs', C=1.0)
lr_gbdt.fit(Xtrain_concat, ytrain)

soft_gbdt = lr_gbdt.predict_proba(Xtest_concat)[:, 1]
ll, auc = evaluate_model(ytest, soft_gbdt, 'GBDT-LR')
results['GBDT-LR'] = {'log_loss': ll, 'roc_auc': auc}

## 8. Model 4 — XGBoost

XGBoost is a highly optimized implementation of gradient boosting. Compared to scikit-learn's `GradientBoostingClassifier`, it:
- Supports **parallelism** (`n_jobs=-1`)
- Uses **column subsampling** and **level-wise pruning** for better regularization
- Is typically **5–10× faster** for the same number of trees

In [ ]:
# Install xgboost if not available
try:
    from xgboost import XGBClassifier
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'xgboost', '-q'])
    from xgboost import XGBClassifier

print('XGBoost ready.')

In [ ]:
xgb = XGBClassifier(
    n_estimators=50,
    learning_rate=0.8,
    max_depth=6,
    n_jobs=-1,          # use all CPU cores
    use_label_encoder=False,
    eval_metric='logloss',
    verbosity=1
)

%time xgb.fit(Xtrain_oh[:, cols_to_keep], ytrain)

soft_xgb = xgb.predict_proba(Xtest_oh[:, cols_to_keep])[:, 1]
ll, auc = evaluate_model(ytest, soft_xgb, 'XGBoost')
results['XGBoost'] = {'log_loss': ll, 'roc_auc': auc}

## 9. Model 5 — Feature Hashing + Random Forest

### 9.1 Merging Rare Modalities

High-cardinality features (e.g., `user` with millions of values) cannot be One-Hot encoded at scale.  
We first **merge rare modalities** (seen fewer than 20 times) into a single `'isRare'` bucket, which:
- Reduces noise from unreliable low-count categories
- Prevents test-set unknown-category issues

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class MergeRareTransformer(BaseEstimator, TransformerMixin):
    """
    Groups infrequent categorical modalities into an 'isRare' bucket.
    Fitted on training data; unseen test values are also mapped to 'isRare'.
    """
    def __init__(self, col_names, threshold):
        self.col_names = col_names
        self.threshold = threshold

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.counts_dict_list_ = []

        for i, col in enumerate(self.col_names):
            counts = X[col].value_counts()
            # Map frequent values to themselves, rare values to 'isRare'
            mapping = {v: ('isRare' if counts[v] < self.threshold[i] else str(v))
                       for v in counts.index}
            self.counts_dict_list_.append(mapping)
        return self

    def transform(self, X):
        Xt = pd.DataFrame()
        for col, mapping in zip(self.col_names, self.counts_dict_list_):
            # Values unseen during fit (new test modalities) are also 'isRare'
            Xt[col] = X[col].apply(lambda x: mapping.get(x, 'isRare'))
        return Xt

In [ ]:
RARE_THRESHOLD = 20

mg = MergeRareTransformer(
    col_names=Xtrain.columns.tolist(),
    threshold=[RARE_THRESHOLD] * len(Xtrain.columns)
)
Xtrain_mg = mg.fit_transform(Xtrain)
Xtest_mg  = mg.transform(Xtest)

print('Cardinality after merging rare modalities:')
print(Xtrain_mg.nunique().sort_values(ascending=False))

n_rare = (Xtrain_mg['app_domain'] == 'isRare').sum()
print(f"\nRare entries in 'app_domain': {n_rare:,}")

### 9.2 Feature Hashing

Instead of building a full vocabulary (which changes across train/test splits), we map each categorical value to an integer using a **hash function** modulo a large number.  

$$\text{hashed\_value} = \text{hash}(\text{string\_value}) \mod 1{,}000{,}000$$

This is the **hashing trick**: it avoids maintaining a category vocabulary and naturally handles new values at test time (they just hash to some slot). The modulo size controls the trade-off between collision rate and memory.

In [ ]:
HASH_MOD = 1_000_000

Xtrain_ha = pd.DataFrame()
Xtest_ha  = pd.DataFrame()

for col in Xtrain_mg.columns:
    # hash() requires a string input; mod ensures the value fits in [0, HASH_MOD)
    Xtrain_ha[col] = Xtrain_mg[col].apply(lambda x: hash(str(x)) % HASH_MOD)
    Xtest_ha[col]  = Xtest_mg[col].apply(lambda x: hash(str(x)) % HASH_MOD)

print('Sample of hashed features:')
Xtrain_ha.head(3)

### 9.3 Random Forest on Hashed Features

Random Forests work directly on integer-valued features and don't require One-Hot Encoding — a key advantage when features have many modalities.  
The hashed integers are treated as ordinal values, and the tree splits naturally discover relevant thresholds.

- `n_estimators=256`: enough trees for stable variance reduction
- `min_samples_leaf=20`: prevents trees from splitting on very rare feature combinations
- `n_jobs=-1`: full parallelism

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=256,
    min_samples_leaf=20,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

%time rf.fit(Xtrain_ha, ytrain)

soft_rf = rf.predict_proba(Xtest_ha)[:, 1]
ll, auc = evaluate_model(ytest, soft_rf, 'Random Forest (hashed)')
results['RF (hashed)'] = {'log_loss': ll, 'roc_auc': auc}

In [ ]:
# Feature importances from the Random Forest
importances = pd.Series(rf.feature_importances_, index=Xtrain_ha.columns)
importances = importances.sort_values(ascending=True)

importances.plot(kind='barh', figsize=(9, 6), color='steelblue', alpha=0.8)
plt.title('Feature Importances — Random Forest')
plt.xlabel('Mean decrease in impurity')
plt.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.show()

### 9.4 XGBoost on Hashed Features

In [ ]:
xgb_ha = XGBClassifier(
    n_estimators=256,
    learning_rate=1.0,
    max_depth=6,
    n_jobs=-1,
    use_label_encoder=False,
    eval_metric='logloss',
    verbosity=0
)

%time xgb_ha.fit(Xtrain_ha, ytrain)

soft_xgb_ha = xgb_ha.predict_proba(Xtest_ha)[:, 1]
ll, auc = evaluate_model(ytest, soft_xgb_ha, 'XGBoost (hashed)')
results['XGBoost (hashed)'] = {'log_loss': ll, 'roc_auc': auc}

## 10. Results Summary & Comparison

We compare all models on two metrics:
- **Log-Loss** (lower is better): measures the quality of predicted probabilities
- **ROC AUC** (higher is better): measures the model's ability to rank clicks above non-clicks, independent of threshold

In [ ]:
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('roc_auc', ascending=False)
results_df.columns = ['Log-Loss ↓', 'ROC AUC ↑']
results_df = results_df.round(5)

print('=== Final Model Comparison ===')
print(results_df.to_string())

# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

results_df['ROC AUC ↑'].sort_values().plot(
    kind='barh', ax=axes[0], color='steelblue', alpha=0.85
)
axes[0].set_title('ROC AUC (higher is better)')
axes[0].set_xlim(0.6, 0.8)
axes[0].grid(axis='x', alpha=0.4)

results_df['Log-Loss ↓'].sort_values(ascending=False).plot(
    kind='barh', ax=axes[1], color='tomato', alpha=0.85
)
axes[1].set_title('Log-Loss (lower is better)')
axes[1].grid(axis='x', alpha=0.4)

plt.suptitle('CTR Prediction — Model Comparison on Avazu (1M samples)', fontsize=13)
plt.tight_layout()
plt.show()

## 11. Conclusion

### Key Takeaways

| Finding | Detail |
|---------|--------|
| **Baseline LR is already decent** | 183 features from low-cardinality columns give a reasonable AUC (~0.67). Simple models on good features go far. |
| **Feature filtering is essential** | With 713k possible OHE features, keeping only the 1,971 seen ≥100 times improves both speed and accuracy. |
| **GBDT-LR captures interactions** | Using GBDT leaf indices as features for LR is a practical way to introduce non-linearities without a complex architecture. |
| **XGBoost is the strongest single model** | Faster than sklearn's GBDT with better regularization. Best results on both OHE and hashed features. |
| **Hashing enables scalability** | Feature hashing makes it possible to train on all high-cardinality features (user, site, app) without memory explosion. |
| **CTR is inherently hard to predict** | AUC ~0.72-0.76 is typical; the best Kaggle submissions for Avazu reach ~0.78 using ensembles and FFM. |

### Possible Extensions
- **Field-aware Factorization Machines (FFM)**: explicitly model pairwise interactions between feature fields
- **Target encoding**: replace categorical values with their mean CTR (with smoothing to avoid leakage)
- **Deep Learning**: embedding layers + MLP (DeepFM, Wide & Deep architecture)
- **Model blending**: average predictions from multiple diverse models for further AUC gains